In [1]:
import re
import json
import torch
import numpy as np
import import_ipynb
from transformers import AutoTokenizer, AutoModel

In [2]:
""" choice of the model:
-PubMedBERT: trained from scratch on PubMed abstracts + full texts
strong on: biomedical terminology, scientific writing
weak on: clinical narrative style (doctor notes, messy reports)
use if: reports are papers/structured findings

-Bio_ClinicalBERT: pretrained on PubMed + fine-tuned on MIMIC-III clinical notes
strong on: real hospital reports, abbreviations, noisy text, shorthand
best balance between biomedical + clinical """

' choice of the model:\n-PubMedBERT: trained from scratch on PubMed abstracts + full texts\nstrong on: biomedical terminology, scientific writing\nweak on: clinical narrative style (doctor notes, messy reports)\nuse if: reports are papers/structured findings\n\n-Bio_ClinicalBERT: pretrained on PubMed + fine-tuned on MIMIC-III clinical notes\nstrong on: real hospital reports, abbreviations, noisy text, shorthand\nbest balance between biomedical + clinical '

In [3]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [5]:
model = AutoModel.from_pretrained(MODEL_NAME)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [6]:
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [7]:
SAVE_PATH = "./bioclinicalbert_local"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bioclinicalbert_local/tokenizer_config.json',
 './bioclinicalbert_local/tokenizer.json')

In [8]:
model = AutoModel.from_pretrained("./bioclinicalbert_local")
tokenizer = AutoTokenizer.from_pretrained("./bioclinicalbert_local")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [10]:
print(f"ClinicalBERT loaded on {device}")

ClinicalBERT loaded on cpu


text to embedding

In [11]:
def encode_report(text: str) -> np.ndarray:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    cls_embedding = outputs.last_hidden_state[:, 0, :]
    
    #emb = outputs.last_hidden_state.mean(dim=1)
    #emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    return cls_embedding.squeeze().cpu().numpy() # return [768] ClinicalBERT CLS embedding

Rule-based NER: extracts tumor entities from report text

In [12]:
# with limitation: no glima will be glioma type, used only because we're 100% that our images contains tumor

TUMOR_TYPES = [
    "glioblastoma", "glioma", "astrocytoma", "meningioma",
    "metastasis", "lymphoma", "medulloblastoma", "lesion", "mass", "tumor"
]

BRAIN_REGIONS = [
    "frontal lobe", "parietal lobe", "temporal lobe", "occipital lobe",
    "cerebellum", "brainstem", "thalamus", "basal ganglia",
    "corpus callosum", "ventricle", "hippocampus", "insula",
    "right frontal", "left frontal", "right parietal", "left parietal",
    "right temporal", "left temporal"
]

ATTRIBUTES = {
    "signal":    r"(high signal|low signal|mixed signal|hyper-?intense|hypo-?intense|iso-?intense)",
    "edema":     r"(edema|oedema|swelling|peritumoral)",
    "necrosis":  r"(necrosis|necrotic|central necrosis)",
    "mass_effect": r"(mass effect|midline shift|herniation|compression|compressive)",
    "enhancement": r"(ring.?enhanc|contrast enhanc|no enhanc|enhanc)",
    "pattern":   r"(speckled|heterogeneous|homogeneous|diffuse|focal|mixed pattern)",
}

In [13]:
def extract_entities(text: str) -> dict:
    # extract tumor type, locations, clinical attributes
    text_lower = text.lower()

    # tumor type
    tumor_type = "unspecified"
    for t in TUMOR_TYPES:
        if t in text_lower:
            tumor_type = t
            break

    # locations
    locations = [r for r in BRAIN_REGIONS if r in text_lower]
    locations = list(dict.fromkeys(locations))

    # laterality
    laterality = "bilateral"
    if "right" in text_lower and "left" not in text_lower:
        laterality = "right"
    elif "left" in text_lower and "right" not in text_lower:
        laterality = "left"

    # attributes
    found_attrs = {}
    for attr_name, pattern in ATTRIBUTES.items():
        matches = re.findall(pattern, text_lower)
        if matches:
            found_attrs[attr_name] = list(set(matches))

    return {
        "tumor_type":  tumor_type,
        "locations":   locations,
        "laterality":  laterality,
        "attributes":  found_attrs,
    }

In [14]:
def generate_prompt(entities: dict, raw_text: str) -> str:
    # convert extracted entities into a structured textual prompt
    loc_str  = ", ".join(entities["locations"]) if entities["locations"] else "unspecified"
    attr_parts = []

    for key, vals in entities["attributes"].items():
        label = key.replace("_", " ")
        attr_parts.append(f"{label}: {', '.join(vals)}")
    attr_str = "; ".join(attr_parts) if attr_parts else "none noted"

    prompt = (
        f"[TUMOR TYPE] {entities['tumor_type']} "
        f"[LATERALITY] {entities['laterality']} "
        f"[LOCATION] {loc_str} "
        f"[ATTRIBUTES] {attr_str} "
        f"[REPORT] {raw_text.strip()}"
    )
    return prompt

In [15]:
def process_report(text: str) -> dict:
    # full pipeline for one report (encode → extract → prompt)
    embedding = encode_report(text)
    entities  = extract_entities(text)
    prompt    = generate_prompt(entities, text)

    return {
        "embedding": embedding,     
        "entities":  entities,      
        "prompt":    prompt,      
    }

In [16]:
def process_all_reports(dataset: list) -> list:
    for i, case in enumerate(dataset):
        if not case.get("label"):
            print(f"[SKIP] {case['id']} has no text report.")
            continue

        print(f"Processing [{i+1}/{len(dataset)}]  {case['id']} ...", end="\r")
        result = process_report(case["label"])

        case["embedding"] = result["embedding"]
        case["entities"]  = result["entities"]
        case["prompt"]    = result["prompt"]

    print(f"\nDone. Processed {len(dataset)} reports.")
    return dataset

test

In [17]:
text = """The lesion area is in the right frontal and parietal lobes with a mixed
pattern of high and low signals with speckled high signal regions. Edema is mainly
observed in the right parietal lobe, partially extending to the frontal lobe,
presenting as high signal, indicating significant tissue swelling around the lesion.
Necrosis is within the lesions of the right parietal and frontal lobes, appearing
as mixed, with alternating high and low signal regions. Ventricular compression is
seen in the lateral ventricles with significant compressive effects on the brain
tissue and ventricles."""

In [18]:
result = process_report(text)

In [19]:
print("entities: ",json.dumps(result["entities"], indent=2))

entities:  {
  "tumor_type": "lesion",
  "locations": [
    "frontal lobe",
    "parietal lobe",
    "ventricle",
    "right frontal",
    "right parietal"
  ],
  "laterality": "right",
  "attributes": {
    "signal": [
      "low signal",
      "high signal"
    ],
    "edema": [
      "edema",
      "swelling"
    ],
    "necrosis": [
      "necrosis"
    ],
    "mass_effect": [
      "compression",
      "compressive"
    ],
    "pattern": [
      "speckled"
    ]
  }
}


In [20]:
print("generated prompt: ",result["prompt"])

generated prompt:  [TUMOR TYPE] lesion [LATERALITY] right [LOCATION] frontal lobe, parietal lobe, ventricle, right frontal, right parietal [ATTRIBUTES] signal: low signal, high signal; edema: edema, swelling; necrosis: necrosis; mass effect: compression, compressive; pattern: speckled [REPORT] The lesion area is in the right frontal and parietal lobes with a mixed
pattern of high and low signals with speckled high signal regions. Edema is mainly
observed in the right parietal lobe, partially extending to the frontal lobe,
presenting as high signal, indicating significant tissue swelling around the lesion.
Necrosis is within the lesions of the right parietal and frontal lobes, appearing
as mixed, with alternating high and low signal regions. Ventricular compression is
seen in the lateral ventricles with significant compressive effects on the brain
tissue and ventricles.


In [21]:
print(f"shape: {result['embedding'].shape}   dtype: {result['embedding'].dtype}")
print(f"sample values: {result['embedding'][:6].round(4)}")

shape: (768,)   dtype: float32
sample values: [-0.0582 -0.2647 -0.0997  0.331   0.1107 -0.2209]


generated prompts for all dataset

In [22]:
import os
import json

In [23]:
OUTPUT_DIR  = "outputs"
PROMPTS_JSON = os.path.join(OUTPUT_DIR, "output.json")
PROMPTS_TXT  = os.path.join(OUTPUT_DIR, "output.txt")

In [24]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [27]:
from ds_loader import load_dataset

/home/amina/miniforge3/envs/biomedclip/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Found 5 cases.
Loading [5/5]  BraTS20_Training_005 ...
Done. Loaded 5 cases.
  total cases   : 5
  unique shapes : {(240, 240, 155)}
  sample labels : ['The lesion area is in the right frontal and parietal lobes with a mixed pattern of high and low signals with speckled high signal regions. Edema is mainly observed in the right parietal lobe, partially extending to the frontal lobe, presenting as high signal, indicating significant tissue swelling around the lesion. Necrosis is within the lesions of the right parietal and frontal lobes, appearing as mixed, with alternating high and low signal regions. Ventricular compression is seen in the lateral ventricles with significant compressive effects on the brain tissue and ventricles.', 'The lesion area is in the right frontal lobe, showing a mixture of high and low signal areas, with some appearing as speckled high signals, and in the right temporal lobe with prominent high-signal areas, some of which are accompanied by low signals, and in

In [29]:
dataset = load_dataset()

Found 5 cases.
Loading [5/5]  BraTS20_Training_005 ...
Done. Loaded 5 cases.


In [31]:
results = []

for i, case in enumerate(dataset):
    if not case.get("label"):
        print(f"[SKIP] {case['id']} — no text report found.")
        continue

    print(f"[{i+1}/{len(dataset)}] processing {case['id']} ...")

    result    = process_report(case["label"])
    entities  = result["entities"]
    prompt    = result["prompt"]

    results.append({
        "id":       case["id"],
        "entities": entities,
        "prompt":   prompt,
    })

[1/5] processing BraTS20_Training_001 ...
[2/5] processing BraTS20_Training_002 ...
[3/5] processing BraTS20_Training_003 ...
[4/5] processing BraTS20_Training_004 ...
[5/5] processing BraTS20_Training_005 ...


In [32]:
with open(PROMPTS_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

In [33]:
with open(PROMPTS_TXT, "w", encoding="utf-8") as f:
    for r in results:
        f.write(f"=== {r['id']} ===\n")
        f.write(r["prompt"] + "\n\n")